# 02 — Application representation, trajectories, and LightGBM

Fit unsupervised reference centroids/scales on pre-cutoff application `h` and monthly `z` embeddings. Score B1 conventional SCR history, B2/B3 static application geometry, B4 reference-month movement and B5 rolling as-of movement. B6 raw embedding coordinates are diagnostic. Each feature table retains every application. Product-specific LightGBM challengers use the same incumbent rows, labels, split and tuning budget. Primary OOT metrics use **observed mature labels only**; inferred labels can be included in training only if `train_provenances` matches the incumbent workflow.

The earlier as-of `h` trajectory requires separately extracted anchor states. The notebook fills missing B5 histories with missing features rather than zero movement. Do not call a trajectory point-in-time valid if its encoder was trained after that anchor.

In [ ]:
from pathlib import Path
import json, sys, shutil
import pandas as pd
REPO = Path.cwd()
if not (REPO / 'scr_features').exists(): REPO = REPO.parent
sys.path.insert(0, str(REPO))
WORK = REPO / 'notebook_runs' / 'demo'  # Switch to 'real' after real notebooks 00–01.
stage00 = json.loads((WORK / 'stage00.json').read_text(encoding='utf-8'))
stage01 = json.loads((WORK / 'stage01.json').read_text(encoding='utf-8'))
DEMO = stage00['demo']
run = stage01['runs'][stage01['chosen']]
print('Frozen temporal model:', stage01['chosen'])

In [ ]:
from scr_features.geometry import fit_geometry, score_geometry, score_rolling_trajectory
from scr_features.hierarchy import encode_applications
from scr_features.features import export_conventional
from scr_features.io import combine_parquet

geometry_dir = WORK / 'models' / 'geometry' / stage01['chosen']
geometry_manifest = fit_geometry(run['fit_applications'], stage00['fit_months'],
                                 stage00['earliest_application'], str(geometry_dir),
                                 clusters=3 if DEMO else 8)
print({k: geometry_manifest[k] for k in ('version', 'fit_max_anchor', 'h_fit_points', 'z_fit_points')})

geometry_files, conventional_files, rolling_files = [], [], []
for group, paths in stage00['groups'].items():
    app_emb = run['scored'][group]
    geom = WORK / 'features' / 'geometry' / f'{group}.parquet'
    conventional = WORK / 'features' / 'conventional' / f'{group}.parquet'
    rolling = WORK / 'features' / 'rolling' / f'{group}.parquet'
    score_geometry(app_emb, paths['months'], str(geometry_dir), str(geom))
    export_conventional(paths['raw'], str(conventional))
    anchor_emb = None
    if group in stage00['anchors']:
        anchor_paths = stage00['anchors'][group]
        anchor_emb = WORK / 'embeddings' / stage01['chosen'] / f'anchor_{group}.parquet'
        encode_applications(anchor_paths['months'], anchor_paths['apps'],
                            run['model'], str(anchor_emb))
    if stage00['anchors']:
        score_rolling_trajectory(str(anchor_emb) if anchor_emb else None,
                                 app_emb, str(geometry_dir), str(rolling))
    geometry_files.append(str(geom))
    conventional_files.append(str(conventional))
    if stage00['anchors']:
        rolling_files.append(str(rolling))

combined_dir = WORK / 'features' / 'combined'
geometry_path = combine_parquet(geometry_files, str(combined_dir / 'geometry.parquet'))
conventional_path = combine_parquet(conventional_files, str(combined_dir / 'conventional.parquet'))
rolling_path = combine_parquet(rolling_files, str(combined_dir / 'rolling.parquet')) if rolling_files else None
geometry_feature_manifest = str(combined_dir / 'geometry.manifest.json')
shutil.copyfile(Path(geometry_files[0]).with_suffix('.manifest.json'), geometry_feature_manifest)
print('Feature files:', geometry_path, conventional_path, rolling_path)

## Locked B0–B6 comparison

`B0` is the incumbent selected feature set. `B1` adds conventional SCR history. `B2` adds static application geometry to B0. `B3` adds it to B1 and is the stronger application-representation test. `B4` and `B5` separately add the two trajectory families to B3. `B6` is a diagnostic raw-coordinate comparison. The three OOT months are evaluated only after the temporal model was selected on train-period validation.

In [ ]:
from scr_features.evaluation import run_ladder

MIN_DELTA_GINI = 0.0 if DEMO else None  # Set a product-approved economic threshold before real OOT evaluation.
DECISION_THRESHOLD = None  # Optionally set the incumbent policy threshold for crossing rates.
if not DEMO and MIN_DELTA_GINI is None:
    raise ValueError('Pre-register MIN_DELTA_GINI before real OOT evaluation')
scorecards = {}
for product in ('CREDIARIO', 'CONSIG_EP'):
    scorecards[product] = run_ladder(
        stage00['incumbent'], conventional_path, run['combined'], geometry_path,
        geometry_feature_manifest, str(WORK / 'evaluation' / product),
        stage00['baseline_columns'], product=product,
        train_provenances=stage00['train_provenances'],
        validation_month=stage00['validation_month'], rolling_path=rolling_path,
        n_bootstrap=10 if DEMO else 200,
        minimum_delta_gini=MIN_DELTA_GINI, decision_threshold=DECISION_THRESHOLD,
        model_params={'n_estimators': 30, 'min_child_samples': 5} if DEMO else None)
summary = pd.DataFrame([
    {'product': product, 'model': model, 'oot_gini': result['oot_observed']['gini'],
     'oot_auc': result['oot_observed']['auc'], 'oot_brier': result['oot_observed']['brier'],
     'paired_delta_gini_ci': result.get('paired_delta_gini_ci')}
    for product, card in scorecards.items() for model, result in card['models'].items()
])
display(summary)
print('Synthetic execution check only' if DEMO else 'Review monthly/segment metrics and economic threshold before any promotion decision')